# DYAMOND Window Extraction Test

1. Load and validate the finalized event catalog
2. Open a DYAMOND dataset and inspect its metadata
3. Investigate how DYAMOND `(x, y)` indices map to geographic coordinates

Do **not** doing bulk extraction yet. There's no reason to do massive data loads just to work out lat/long mapping and time alignment from DYAMOND to IBTrACS.

In [1]:
import sys
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    required = {"data", "src", "notebooks", "outputs", "README.md"}
    for candidate in candidates:
        names = {p.name for p in candidate.iterdir()} if candidate.exists() else set()
        if required.issubset(names):
            return candidate
    raise FileNotFoundError("Could not locate project root from current working directory.")
    
PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

WINDOWS_DIR = PROCESSED_DIR / "windows"
METADATA_DIR = PROCESSED_DIR / "window_metadata"

for path in [WINDOWS_DIR, METADATA_DIR, OUTPUTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/home/exouser/SciVis_2026')

In [2]:
from src.io.dyamond import (
    DYAMOND_BASE_URL,
    available_variables,
    dataset_summary,
    get_dataset_spec,
    open_dataset,
    read_data,
)

In [3]:
CONFIG = {
    # from storm selection
    "event_catalog_path": PROCESSED_DIR / "event_catalog.parquet",
    "test_event_row": 0,
    # pick one DYAMOND variable to check metadata and grid mapping
    "mapping_probe_var": "salt",
    "sample_time_index": 0,
    "sample_quality": -18,
    "sample_z": [0, 1],
    "max_allowed_time_error_hours": 3,
}

CONFIG

{'event_catalog_path': PosixPath('/home/exouser/SciVis_2026/data/processed/event_catalog.parquet'),
 'test_event_row': 0,
 'mapping_probe_var': 'salt',
 'sample_time_index': 0,
 'sample_quality': -18,
 'sample_z': [0, 1],
 'max_allowed_time_error_hours': 3}

## Load and validate event catalog
check the following:
- `event_id`
- `SID`
- `anchor_time`
- `box_size_deg`
- `pre_hours`

In [4]:
EVENT_CATALOG_PATH = Path(CONFIG["event_catalog_path"])

if not EVENT_CATALOG_PATH.exists():
    raise FileNotFoundError(
        f"Event catalog not found: {EVENT_CATALOG_PATH}\n"
        "Make sure to run the storm selection notebook first and save data/processed/event_catalog.parquet."
    )

event_catalog = pd.read_parquet(EVENT_CATALOG_PATH)
print("Loaded event catalog:", EVENT_CATALOG_PATH)
print("Rows:", len(event_catalog))
event_catalog.head()

Loaded event catalog: /home/exouser/SciVis_2026/data/processed/event_catalog.parquet
Rows: 8


,event_id,priority_order,SID,name,season,basin,start_time,end_time,anchor_time,anchor_type,duration_hours,n_points,max_wind_kt,min_pres_mb,min_lat,max_lat,min_lon_180,max_lon_180,box_size_deg,pre_hours,post_hours,precursor_offsets_hours,selection_reason,notes
0,event_01_2020034S17129,1,2020034S17129,DAMIEN,2020,SI,2020-02-03 00:00:00+00:00,2020-02-10 12:00:00+00:00,2020-02-08 06:00:00+00:00,max_wind,180.0,67,95.0,956.0,-25.7,-16.0,116.4,129.3,15,48,0,"6,12,24,48",None,None
1,event_02_2020092S09155,2,2020092S09155,HAROLD,2020,SP,2020-04-01 00:00:00+00:00,2020-04-11 00:00:00+00:00,2020-04-06 06:00:00+00:00,max_wind,240.0,81,145.0,909.0,-35.0,-9.2,-179.1,179.9,15,48,0,"6,12,24,48",None,None
2,event_03_2020244N25146,3,2020244N25146,HAISHEN,2020,WP,2020-08-30 12:00:00+00:00,2020-09-10 06:00:00+00:00,2020-09-04 06:00:00+00:00,max_wind,258.0,87,135.0,913.0,19.3,45.9,124.0,146.4,15,48,0,"6,12,24,48",None,None
3,event_04_2020251N17319,4,2020251N17319,PAULETTE,2020,None,2020-09-07 00:00:00+00:00,2020-09-28 12:00:00+00:00,2020-09-14 18:00:00+00:00,max_wind,516.0,174,90.0,965.0,16.9,47.1,-64.9,-16.6,15,48,0,"6,12,24,48",None,None
4,event_05_2020299N11144,5,2020299N11144,GONI,2020,WP,2020-10-25 06:00:00+00:00,2020-11-06 06:00:00+00:00,2020-10-31 18:00:00+00:00,max_wind,288.0,97,170.0,884.0,10.6,16.7,107.9,143.9,15,48,0,"6,12,24,48",None,None


In [5]:
required_event_cols = ["event_id", "SID", "anchor_time", "box_size_deg", "pre_hours"]

missing_cols = [c for c in required_event_cols if c not in event_catalog.columns]
if missing_cols:
    raise ValueError(
        "Event catalog is missing required columns for extraction: "
        f"{missing_cols}"
    )

# fix datetimes
for col in ["start_time", "end_time", "anchor_time"]:
    if col in event_catalog.columns:
        event_catalog[col] = pd.to_datetime(event_catalog[col], utc=True, errors="coerce")

checks = {
    "n_events": len(event_catalog),
    "unique_event_id": event_catalog["event_id"].nunique(),
    "unique_sid": event_catalog["SID"].nunique(),
    "anchor_time_notna_frac": float(event_catalog["anchor_time"].notna().mean()),
    "box_size_deg_notna_frac": float(event_catalog["box_size_deg"].notna().mean()),
    "pre_hours_notna_frac": float(event_catalog["pre_hours"].notna().mean()),
}

print("Event catalog validation summary:")
pprint(checks)

if (event_catalog["event_id"].nunique() != len(event_catalog)):
    raise ValueError("event_id must be unique per event.")

Event catalog validation summary:
{'anchor_time_notna_frac': 1.0,
 'box_size_deg_notna_frac': 1.0,
 'n_events': 8,
 'pre_hours_notna_frac': 1.0,
 'unique_event_id': 8,
 'unique_sid': 8}


In [6]:
display_cols = [
    c for c in [
        "event_id",
        "priority_order",
        "SID",
        "name",
        "season",
        "basin",
        "start_time",
        "end_time",
        "anchor_time",
        "anchor_type",
        "duration_hours",
        "max_wind_kt",
        "min_pres_mb",
        "min_lat",
        "max_lat",
        "min_lon_180",
        "max_lon_180",
        "box_size_deg",
        "pre_hours",
        "post_hours",
        "precursor_offsets_hours",
    ]
    if c in event_catalog.columns
]

event_catalog[display_cols].sort_values("anchor_time").reset_index(drop=True)

,event_id,priority_order,SID,name,season,basin,start_time,end_time,anchor_time,anchor_type,duration_hours,max_wind_kt,min_pres_mb,min_lat,max_lat,min_lon_180,max_lon_180,box_size_deg,pre_hours,post_hours,precursor_offsets_hours
0,event_01_2020034S17129,1,2020034S17129,DAMIEN,2020,SI,2020-02-03 00:00:00+00:00,2020-02-10 12:00:00+00:00,2020-02-08 06:00:00+00:00,max_wind,180.0,95.0,956.0,-25.7,-16.0,116.4,129.3,15,48,0,"6,12,24,48"
1,event_02_2020092S09155,2,2020092S09155,HAROLD,2020,SP,2020-04-01 00:00:00+00:00,2020-04-11 00:00:00+00:00,2020-04-06 06:00:00+00:00,max_wind,240.0,145.0,909.0,-35.0,-9.2,-179.1,179.9,15,48,0,"6,12,24,48"
2,event_03_2020244N25146,3,2020244N25146,HAISHEN,2020,WP,2020-08-30 12:00:00+00:00,2020-09-10 06:00:00+00:00,2020-09-04 06:00:00+00:00,max_wind,258.0,135.0,913.0,19.3,45.9,124.0,146.4,15,48,0,"6,12,24,48"
3,event_04_2020251N17319,4,2020251N17319,PAULETTE,2020,None,2020-09-07 00:00:00+00:00,2020-09-28 12:00:00+00:00,2020-09-14 18:00:00+00:00,max_wind,516.0,90.0,965.0,16.9,47.1,-64.9,-16.6,15,48,0,"6,12,24,48"
4,event_05_2020299N11144,5,2020299N11144,GONI,2020,WP,2020-10-25 06:00:00+00:00,2020-11-06 06:00:00+00:00,2020-10-31 18:00:00+00:00,max_wind,288.0,170.0,884.0,10.6,16.7,107.9,143.9,15,48,0,"6,12,24,48"
5,event_06_2020306N15288,6,2020306N15288,ETA,2020,None,2020-10-31 18:00:00+00:00,2020-11-14 00:00:00+00:00,2020-11-03 00:00:00+00:00,max_wind,318.0,130.0,922.0,13.6,35.7,-87.8,-69.2,15,48,0,"6,12,24,48"
6,event_07_2021035S11080,7,2021035S11080,FARAJI,2021,SI,2021-02-04 00:00:00+00:00,2021-02-16 06:00:00+00:00,2021-02-08 18:00:00+00:00,max_wind,294.0,140.0,920.0,-20.3,-10.6,66.2,85.3,15,48,0,"6,12,24,48"
7,event_08_2021062S14064,8,2021062S14064,HABANA,2021,SI,2021-03-02 18:00:00+00:00,2021-03-17 12:00:00+00:00,2021-03-10 12:00:00+00:00,max_wind,354.0,135.0,928.0,-23.7,-14.0,64.2,81.1,15,48,0,"6,12,24,48"


In [7]:
test_idx = int(CONFIG["test_event_row"])

if not (0 <= test_idx < len(event_catalog)):
    raise IndexError(
        f"CONFIG['test_event_row']={test_idx} is out of range for "
        f"{len(event_catalog)} events."
    )

test_event = event_catalog.iloc[[test_idx]].copy().reset_index(drop=True)
test_event_row = test_event.iloc[0]
print("Selected test event:")
display(test_event)

Selected test event:


,event_id,priority_order,SID,name,season,basin,start_time,end_time,anchor_time,anchor_type,duration_hours,n_points,max_wind_kt,min_pres_mb,min_lat,max_lat,min_lon_180,max_lon_180,box_size_deg,pre_hours,post_hours,precursor_offsets_hours,selection_reason,notes
0,event_01_2020034S17129,1,2020034S17129,DAMIEN,2020,SI,2020-02-03 00:00:00+00:00,2020-02-10 12:00:00+00:00,2020-02-08 06:00:00+00:00,max_wind,180.0,67,95.0,956.0,-25.7,-16.0,116.4,129.3,15,48,0,"6,12,24,48",None,None


In [8]:
def parse_precursor_offsets(value) -> list[int]:
    """
    event-catalog precursor offset should become list of sorted hours
    """
    if pd.isna(value):
        return []
    if isinstance(value, str):
        parts = [p.strip() for p in value.split(",") if p.strip()]
        return sorted(int(p) for p in parts)
    if isinstance(value, (list, tuple, np.ndarray)):
        return sorted(int(v) for v in value)
    raise TypeError(f"Unsupported precursor offset format: {type(value)!r}")

test_event_summary = {
    "event_id": test_event_row["event_id"],
    "SID": test_event_row["SID"],
    "anchor_time_utc": test_event_row["anchor_time"],
    "box_size_deg": test_event_row["box_size_deg"],
    "pre_hours": test_event_row["pre_hours"],
    "precursor_offsets_hours": parse_precursor_offsets(
        test_event_row.get("precursor_offsets_hours", pd.NA)
    ),
}

pprint(test_event_summary)

{'SID': '2020034S17129',
 'anchor_time_utc': Timestamp('2020-02-08 06:00:00+0000', tz='UTC'),
 'box_size_deg': 15,
 'event_id': 'event_01_2020034S17129',
 'pre_hours': 48,
 'precursor_offsets_hours': [6, 12, 24, 48]}


## Open one DYAMOND dataset and inspect metadata

Inspect remote dataset metadata and record:
- logic box
- shape
- field name
- timestep count
- available timestep indices

No grid mapping has happened yet, so code can't assume:
- a regular lat lon grid
- a specific time interval
- that index space is already geographic

In [9]:
probe_var = CONFIG["mapping_probe_var"]

print("DYAMOND base URL:", DYAMOND_BASE_URL)
print("Variables available in current dyamond.py:", available_variables())
print("Selected probe variable:", probe_var)

probe_spec = get_dataset_spec(probe_var)
print("Dataset spec:")
pprint({
    "variable": probe_spec.variable,
    "url": probe_spec.url,
})

DYAMOND base URL: https://nsdf-climate3-origin.nationalresearchplatform.org:50098/nasa/nsdf/climate3/dyamond/
Variables available in current dyamond.py: ['salt', 'v', 'theta', 'w', 'u']
Selected probe variable: salt
Dataset spec:
{'url': 'https://nsdf-climate3-origin.nationalresearchplatform.org:50098/nasa/nsdf/climate3/dyamond/mit_output/llc2160_salt/salt_llc2160_x_y_depth.idx',
 'variable': 'salt'}


In [10]:
# probe_db = open_dataset(probe_var)
# probe_meta = dataset_summary(probe_db)
# probe_print = probe_meta.copy()
# ts = probe_print.get("timesteps", [])
# probe_print["timesteps"] = {
#     "count": len(ts),
#     "first": ts[0] if len(ts) > 0 else None,
#     "last": ts[-1] if len(ts) > 0 else None,
# }
# print("Remote dataset summary:")
# pprint(probe_print)

In [12]:
probe_db = open_dataset(probe_var)
probe_meta = dataset_summary(probe_db)
logic_box = probe_meta["logic_box"]
shape = probe_meta["shape"]
timesteps = probe_meta["timesteps"]

print("Logic box:", logic_box)
print("Shape:", shape)
print("Number of dimensions:", probe_meta["dims"])
print("Number of timesteps:", probe_meta["n_timesteps"])

if len(timesteps) > 0:
    print("First timestep index:", timesteps[0])
    print("Last timestep index:", timesteps[-1])
else:
    print("No timesteps reported.")

Logic box: ((0, 0, 0), (8640, 6480, 90))
Shape: (8640, 6480, 90)
Number of dimensions: 3
Number of timesteps: 10366
First timestep index: 0
Last timestep index: 10365


In [13]:
sample_arr = read_data(
    probe_db,
    time=CONFIG["sample_time_index"],
    quality=CONFIG["sample_quality"],
    z=CONFIG["sample_z"],
)

print("Sample array type:", type(sample_arr))
print("Sample array shape:", getattr(sample_arr, "shape", None))
print("Sample array dtype:", getattr(sample_arr, "dtype", None))

Sample array type: <class 'numpy.ndarray'>
Sample array shape: (1, 102, 135)
Sample array dtype: float32


In [14]:
dataset_metadata_table = pd.DataFrame([{
    "variable": probe_var,
    "url": probe_meta["url"],
    "dims": probe_meta["dims"],
    "shape": str(probe_meta["shape"]),
    "logic_box": str(probe_meta["logic_box"]),
    "n_timesteps": probe_meta["n_timesteps"],
    "first_timestep": timesteps[0] if len(timesteps) > 0 else pd.NA,
    "last_timestep": timesteps[-1] if len(timesteps) > 0 else pd.NA,
    "field_name": probe_meta["field_name"],
}])

dataset_metadata_table

,variable,url,dims,shape,logic_box,n_timesteps,first_timestep,last_timestep,field_name
0,salt,https://nsdf-climate3-origin.nationalresearchp...,3,"(8640, 6480, 90)","((0, 0, 0), (8640, 6480, 90))",10366,0,10365,salt


## Investigate grid mapping

Must figure out how DYAMOND `(x, y)` maps to geographic
coordinates before converting event lat lon positions into index windows

1. inspect what metadata and interfaces are available from the OpenVisus dataset object
2. fail explicitly if a geographic mapping source has not yet been identified

In [17]:
def safe_getattr(obj, name, default=None):
    try:
        return getattr(obj, name)
    except Exception:
        return default

db_methods = sorted(
    name for name in dir(probe_db)
    if not name.startswith("_")
)

print("Some dataset attributes/methods:")
for name in db_methods[:120]:
    print(name)

Some dataset attributes/methods:
compressDataset
compress_dataset
copyBlocks
copyDataset
copyDatasetToCloud
createAccess
createAccessForBlockQuery
db
field_names
getBounds
getExtendedInfo
getField
getFields
getLogicBox
getLogicSize
getMaxResolution
getPointDim
getSliceLogicBox
getTimesteps
getXSlice
getXYSlice
max_resolution
read
readBlock
shape
write
writeBlock
writeSlabs


In [18]:
candidate_method_names = [
    "getField",
    "getFields",
    "getLogicBox",
    "getTimesteps",
    "getDatasetBody",
    "getPhysicBox",
    "getAxis",
    "getAxes",
    "getBounds",
    "getPointDim",
]

available_candidates = {
    name: hasattr(probe_db, name) for name in candidate_method_names
}

print("Potentially relevant methods for spatial interpretation:")
pprint(available_candidates)

Potentially relevant methods for spatial interpretation:
{'getAxes': False,
 'getAxis': False,
 'getBounds': True,
 'getDatasetBody': True,
 'getField': True,
 'getFields': True,
 'getLogicBox': True,
 'getPhysicBox': False,
 'getPointDim': True,
 'getTimesteps': True}


In [20]:
field_obj = probe_db.getField()
print("Field object:", field_obj)

field_attrs = [name for name in dir(field_obj) if not name.startswith("_")]
print("\nField attributes/methods:")
for name in field_attrs[:120]:
    print(name)

Field object: <VisusKernelPy.Field; proxy of <Swig Object of type 'Visus::Field *' at 0x72e3b86e7c90> >

Field attributes/methods:
default_compression
default_layout
default_value
description
dtype
filter
fromString
getDTypeRange
getDescription
getParam
hasParam
index
name
params
read
setDTypeRange
setDescription
this
thisown
toString
valid
write


In [21]:
# inspect any spatial metadata if dataset exposes it
extra_spatial_info = {}

if hasattr(probe_db, "getPhysicBox"):
    try:
        extra_spatial_info["physic_box"] = probe_db.getPhysicBox()
    except Exception as exc:
        extra_spatial_info["physic_box_error"] = repr(exc)

if hasattr(probe_db, "getDatasetBody"):
    try:
        extra_spatial_info["dataset_body"] = str(probe_db.getDatasetBody())[:1000]
    except Exception as exc:
        extra_spatial_info["dataset_body_error"] = repr(exc)

pprint(extra_spatial_info)

{'dataset_body': '<VisusKernelPy.StringTree; proxy of <Swig Object of type '
                 "'std::shared_ptr< Visus::StringTree > *' at 0x72e3ed7851a0> "
                 '>'}


In [22]:
grid_mapping_notes = {
    "probe_variable": probe_var,
    "logic_box": probe_meta["logic_box"],
    "shape": probe_meta["shape"],
    "mapping_source_identified": False,
    "mapping_source_type": pd.NA,
    "mapping_source_details": pd.NA,
    "status": (
        "UNRESOLVED: current project files do not yet confirm how DYAMOND x/y "
        "indices map to latitude/longitude. Do not proceed to lat/lon-based extraction."
    ),
}

grid_mapping_notes_df = pd.DataFrame([grid_mapping_notes])
grid_mapping_notes_df

,probe_variable,logic_box,shape,mapping_source_identified,mapping_source_type,mapping_source_details,status
0,salt,"((0, 0, 0), (8640, 6480, 90))","(8640, 6480, 90)",False,<NA>,<NA>,UNRESOLVED: current project files do not yet c...


In [23]:
# save discovery note so later notebook iterations have a record
# can be useful for tracking if changes are improving mapping :(
grid_mapping_note_path = METADATA_DIR / "grid_mapping_probe_status.parquet"
grid_mapping_notes_df.to_parquet(grid_mapping_note_path, index=False)

print("Saved:", grid_mapping_note_path)

Saved: /home/exouser/SciVis_2026/data/processed/window_metadata/grid_mapping_probe_status.parquet
